# FC Mold G5 - Pipeline Tests

Smoke tests and validation for the `src/` modules.
Run after making changes to verify nothing is broken.

In [0]:
import sys
project_root = "/Workspace/Users/dieudonne.nkulikiyimfura@se.abb.com/TATAIjmulden_FCMoldG5"
if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [0]:
from src.config import CONFIG, STRAND_CONFIGS, METADATA_PATH

assert CONFIG.window_size == 300, "Default window_size should be 300"
assert CONFIG.vc_threshold == 0.1
assert CONFIG.ml_stability_threshold_mm == 2.0
assert "23_5" in STRAND_CONFIGS
assert "23_6" in STRAND_CONFIGS
assert STRAND_CONFIGS["23_6"].strand_name == "Strand 23-6"
print("PASS: Config imports and defaults")

In [0]:
import numpy as np
from src.disturbance_detection import (
    detect_excursion_event_robust,
    detect_slow_drift_event,
    detect_transient_bump_dynamic,
    detect_high_variability_event,
)

# Stable signal: should NOT trigger any detector
stable = np.random.normal(0, 0.5, 300)
assert not detect_excursion_event_robust(stable, threshold_mm=8.0)
assert not detect_high_variability_event(stable, ptp_threshold_mm=10.0)
print("PASS: Stable signal triggers no disturbances")

# Excursion signal: large spike
excursion = np.zeros(300)
excursion[100:120] = 15.0  # 20-sample spike at 15 mm
assert detect_excursion_event_robust(excursion, threshold_mm=8.0, min_duration_s=5.0)
print("PASS: Excursion detected on synthetic spike")

# High variability signal
noisy = np.random.normal(0, 6, 300)
assert detect_high_variability_event(noisy, ptp_threshold_mm=10.0)
print("PASS: High variability detected on noisy signal")

In [0]:
import pandas as pd
from src.sequence_analysis import (
    custom_rounding,
    segment_by_time_gaps,
    identify_sequences,
)

# Create synthetic casting data: constant speed
n = 600
df_test = pd.DataFrame({
    "plainTimeStamp": pd.date_range("2025-04-01", periods=n, freq="1s"),
    "castingSpeed": [1.2] * n,
    "EMBR_col": [100.0] * n,
})

normal, abnormal = identify_sequences(
    df_test, Vc_column="castingSpeed", window_size=300, Vc_threshold=0.1
)

assert len(normal) == 2, f"Expected 2 normal sequences, got {len(normal)}"
assert len(abnormal) == 0, f"Expected 0 abnormal, got {len(abnormal)}"
print(f"PASS: Sequence identification ({len(normal)} normal, {len(abnormal)} abnormal)")

# Test segmentation with a gap
df_gap = df_test.copy()
df_gap.loc[300:, "plainTimeStamp"] += pd.Timedelta("1h")  # 1-hour gap
df_seg = segment_by_time_gaps(df_gap, max_gap_seconds=5)
assert df_seg["segment_id"].nunique() == 2, "Should detect 2 segments"
print("PASS: Time gap segmentation")

In [0]:
from src.data_loading import (
    add_plain_timestamp,
    get_expert_files,
    load_expert_files,
    convert_units,
    StrandDataLoader,
)
from src.pipeline import StrandAnalysisPipeline

print("PASS: All modules import successfully")

In [0]:
# Full integration test - runs the actual pipeline on real data
# Uncomment to run (takes ~2-5 minutes):

# pipeline = StrandAnalysisPipeline(STRAND_CONFIGS["23_6"], CONFIG, spark, dbutils)
# results = pipeline.run()
# assert results["success"], f"Pipeline failed: {results.get('error')}"
# assert len(results["df_seq"]) > 50, "Expected >50 sequences"
# assert "disturbance_type" in results["df_seq"].columns
# print(f"PASS: End-to-end pipeline ({len(results['df_seq'])} sequences)")

In [0]:
print("\n" + "="*50)
print("ALL TESTS PASSED")
print("="*50)